## This script skletonizes all the masks of all the .tiff files in a folder and saves the xy coordinate of the skleton in results/results.csv file in the same directory as the tiff files.
The coordinate of the points in results csv are later used in ImportMasksToDLC.ipynb script to import in DLC and so on

In [3]:
# First cell: import necessary modules
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import pickle
import time

from backbone import Backbone
from candidate_point import CandidatePoints
from candidate_point_detect import CandidatePointsDetect
from constant import WORM, ROOT_SMOOTH
from graph import Graph
from graph_builder import GraphBuilder
from graph_prune import GraphPrune
from root_smooth import RootSmooth
from search_backbone import SearchBackbone
from skimage import io, color, morphology, measure
from scipy.ndimage import distance_transform_edt
from skimage import color, measure, morphology
from scipy.ndimage import distance_transform_edt
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
import pickle

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Second cell: utility functions
def load_skip_list(filename):
    skip_list = []
    try:
        with open(filename, 'r') as file:
            for line in file:
                skip_list.append(int(line.strip()))
    except IOError:
        pass
    return skip_list

def image_get(file_path, img_index):
    # First try the filename with zfill(5)
    image_filename_zfill = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{img_index}.png"
    print(image_filename_zfill)
    if os.path.exists(image_filename_zfill):
        image_filename = image_filename_zfill
    elif not os.path.exists(image_filename):
        print(f"Pic {img_index} : Not Exist!")
        return None
    
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        print(f"Pic {img_index} : Failed to Load!")
        return None

    #gray = (image < 163) & (image > 161)
    #image = measure.label(gray).astype(np.uint8)
    
    if image.shape[0] != WORM.IMAGE_SIZE2 or image.shape[1] != WORM.IMAGE_SIZE1:
        print(f"Pic {img_index} : Wrong Image Size!")
        return None
    
    return image

def image_get2(file_path, img_index):
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    gray =  (image < 179) & (image > 177)
    image = measure.label(gray).astype(np.uint8)
    
    if image is None or image.shape[0] != WORM.IMAGE_SIZE2 or image.shape[1] != WORM.IMAGE_SIZE1:
        print(f"Pic {img_index} : Not Exist or Wrong Image Size 1!")
        return None
    return image

def image_get1(file_path, img_index):
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        print(f"Pic {img_index} : Not Exist!")
        return None
    
    # Assuming we want to take the left half of the image
    left_half_image = image[:, :image.shape[1] // 2]
    
    gray = (left_half_image < 163) & (left_half_image > 161)
    labeled_image = measure.label(gray).astype(np.uint8)

    
    return labeled_image

def in_skip_list(index, skip_list):
    return index in skip_list


# Save results to CSV
def save_to_csv(save_path, image_path, ordered_points, max_thickness, length):
    frame_number = os.path.basename(image_path).split('.')[0].replace('frame', '')
    filename = os.path.join(save_path, "results.csv")
    ordered_points_flattened = [coord for point in ordered_points for coord in point]

    with open(filename, 'a') as file:
        file.write(','.join([frame_number] + [str(max_thickness)] + [str(length)] + list(map(str, ordered_points_flattened))) + '\n')



# Example gradient function
def interpolate_color(start_color, end_color, factor):
    return tuple([start_color[i] + factor * (end_color[i] - start_color[i]) for i in range(3)])

# Normalize the color values to 0-1 range for matplotlib
def normalize_color(color):
    return [c / 255.0 for c in color]

In [5]:
# define the main processing function
pic_start =0
pic_end = 12627#14400#12748

start_color = (255, 0, 0) # Red
end_color = (0, 0, 255)  # Blue


PIC_START = pic_start
PIC_END = pic_end


def process_folder(folder):
    search_backbone = SearchBackbone()
    A = folder#+'_masks'
    print(A)
    file_path = os.path.join(folder, A)
    save_folder = os.path.join(folder, 'results')
    save_folder_BBones = os.path.join(folder, 'backbones')
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    if not os.path.exists(save_folder_BBones):
        os.makedirs(save_folder_BBones)    

    for pic_num in range(PIC_START, PIC_END + 1):
        image = image_get(file_path, pic_num)
        if image is not None:
            filename = f"{save_folder_BBones}/backbone_{pic_num}.bin"

            try:
                start = time.time()
                backbone = search_backbone.search(image)
                end = time.time()
                print(f"Centerline extraction time consumption: {(end - start) * 1000}ms {pic_num}")

                distance_transform = distance_transform_edt(image)
                thicknesses = [distance_transform[int(y), int(x)] for y, x in search_backbone.temp_backbone.cood]
                max_thickness = max(thicknesses)
                length = search_backbone.temp_backbone.wormLength
                save_to_csv(save_folder, f"{file_path}/frame{pic_num}.tiff", search_backbone.temp_backbone.cood, max_thickness, length)

            except Exception as e:
                print(f"Error: {e}")
            search_backbone.save_centerline_results(filename)

In [7]:
pic_start =0
pic_end = 12627#12748

start_color = (255, 0, 0)  # Red
end_color = (90, 0, 255)  # Blue


PIC_START = pic_start
PIC_END = pic_end


# List of folders to process
folders = [
    '/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650'  
]
for folder in folders:
    process_folder(folder)

/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_0_label.tiff
Normal
145
Current backbone length: 290.44911598453507, Mean length: 0.0
Centerline extraction time consumption: 275.56467056274414ms 0
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_1_label.tiff
Omega
189
Current backbone length: 355.53896071800665, Mean length: 290.44911598453507
Centerline extraction time consumption: 341.92442893981934ms 1
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_2_label.tiff
Delta
175
Current backbone length: 353.26771759797475, Mean length: 290.44911598453507
Centerline extraction time consumption: 325.0744342803955ms 2
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_3_label.tiff
Omega
184
Current backbone length: 357.6757394587457, Mean length: 290.44911598453507
Cen

Omega
199
Current backbone length: 358.8037529330206, Mean length: 290.44911598453507
Centerline extraction time consumption: 353.1780242919922ms 23
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_24_label.tiff
Omega
80
Current backbone length: 154.2082558165741, Mean length: 290.44911598453507
Centerline extraction time consumption: 321.00367546081543ms 24
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_25_label.tiff
Delta
176
Current backbone length: 354.48349006445846, Mean length: 290.44911598453507
Centerline extraction time consumption: 303.68757247924805ms 25
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_26_label.tiff
Omega
198
Current backbone length: 355.6073746926614, Mean length: 290.44911598453507
Centerline extraction time consumption: 360.368013381958ms 26
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_27_label.tiff
Ome

Omega
161
Current backbone length: 331.32084212721173, Mean length: 309.24561465786724
Centerline extraction time consumption: 358.54434967041016ms 53
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_54_label.tiff
Omega
161
Current backbone length: 329.7774687321456, Mean length: 312.3992185820593
Centerline extraction time consumption: 295.316219329834ms 54
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_55_label.tiff
Omega
186
Current backbone length: 338.2943528666789, Mean length: 314.5714998508201
Centerline extraction time consumption: 357.8076362609863ms 55
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_56_label.tiff
101
70
11
70
Omega
70
Current backbone length: 116.0934815136339, Mean length: 317.2073724081377
Centerline extraction time consumption: 322.48616218566895ms 56
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_57_labe

Omega
151
Current backbone length: 325.89197571825076, Mean length: 329.21483944231875
Centerline extraction time consumption: 300.0051975250244ms 86
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_87_label.tiff
Omega
164
Current backbone length: 342.8746392072414, Mean length: 329.1296378083683
Centerline extraction time consumption: 357.6321601867676ms 87
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_88_label.tiff
Omega
168
Current backbone length: 338.35393161842546, Mean length: 329.4732628433401
Centerline extraction time consumption: 336.9028568267822ms 88
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_89_label.tiff
Omega
168
Current backbone length: 340.1745021066781, Mean length: 329.68986452078116
Centerline extraction time consumption: 371.77586555480957ms 89
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_90_label.tiff
Ome

Omega
115
Current backbone length: 243.79301286265917, Mean length: 332.0773735429744
Centerline extraction time consumption: 338.1679058074951ms 120
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_121_label.tiff
Omega
159
Current backbone length: 308.5348618407316, Mean length: 332.0773735429744
Centerline extraction time consumption: 314.54944610595703ms 121
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_122_label.tiff
101
40
11
40
Omega
40
Current backbone length: 80.68228318291395, Mean length: 331.6976556122931
Centerline extraction time consumption: 260.2503299713135ms 122
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_123_label.tiff
Omega
95
Current backbone length: 198.05604893245516, Mean length: 331.6976556122931
Centerline extraction time consumption: 256.4713954925537ms 123
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_1

...

Omega
141
Current backbone length: 294.40498777536914, Mean length: 346.6790603654056
Centerline extraction time consumption: 280.9433937072754ms 12489
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12490_label.tiff
Omega
115
Current backbone length: 243.21035537862295, Mean length: 346.6790603654056
Centerline extraction time consumption: 286.17072105407715ms 12490
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12491_label.tiff
Omega
122
Current backbone length: 271.44529792625275, Mean length: 346.6790603654056
Centerline extraction time consumption: 301.1810779571533ms 12491
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12492_label.tiff
Omega
110
Current backbone length: 226.43310703310723, Mean length: 346.6790603654056
Centerline extraction time consumption: 232.283353805542ms 12492
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/fra

101
186
11
186
Delta
186
Current backbone length: 347.48616678569795, Mean length: 346.68693296814746
Centerline extraction time consumption: 299.29542541503906ms 12515
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12516_label.tiff
Error: Prune Error!!! Special Node Num Must Be 2!!!
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12517_label.tiff
101
89
11
89
Omega
89
Current backbone length: 158.76176949789297, Mean length: 346.687060763769
Centerline extraction time consumption: 133.5766315460205ms 12517
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12518_label.tiff
Omega
153
Current backbone length: 305.506005664671, Mean length: 346.687060763769
Centerline extraction time consumption: 279.1025638580322ms 12518
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12519_label.tiff
Omega
111
Current backbone length: 203.7942549778261, M

Omega
163
Current backbone length: 316.17548958933816, Mean length: 346.68041674433096
Centerline extraction time consumption: 312.94941902160645ms 12540
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12541_label.tiff
101
175
11
175
Omega
175
Current backbone length: 333.45998823029817, Mean length: 346.67554375277257
Centerline extraction time consumption: 343.33181381225586ms 12541
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12542_label.tiff
101
110
11
110
Omega
110
Current backbone length: 236.75528328314198, Mean length: 346.6734329788511
Centerline extraction time consumption: 355.5312156677246ms 12542
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12543_label.tiff
101
154
11
154
Delta
154
Current backbone length: 305.42947055006346, Mean length: 346.6734329788511
Centerline extraction time consumption: 346.0431098937988ms 12543
/mnt/DATA/Mahsa/movies/LongRecordi

Omega
167
Current backbone length: 340.06572139822924, Mean length: 346.63726107603617
Centerline extraction time consumption: 358.11758041381836ms 12567
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12568_label.tiff
Omega
165
Current backbone length: 341.28753375968074, Mean length: 346.636213818717
Centerline extraction time consumption: 414.33024406433105ms 12568
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12569_label.tiff
Omega
162
Current backbone length: 348.954220945491, Mean length: 346.6353615752404
Centerline extraction time consumption: 434.37647819519043ms 12569
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12570_label.tiff
Omega
168
Current backbone length: 362.34587057163435, Mean length: 346.63573099683833
Centerline extraction time consumption: 441.04719161987305ms 12570
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/

Omega
154
Current backbone length: 354.5550471758996, Mean length: 346.6464090162542
Centerline extraction time consumption: 328.6721706390381ms 12598
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12599_label.tiff
101
83
11
83
Omega
83
Current backbone length: 174.0427299869605, Mean length: 346.64766395725064
Centerline extraction time consumption: 328.26733589172363ms 12599
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12600_label.tiff
Omega
161
Current backbone length: 347.67878982550957, Mean length: 346.64766395725064
Centerline extraction time consumption: 311.50221824645996ms 12600
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_09172024151650/frame_12601_label.tiff
Normal
175
Current backbone length: 358.2945014155512, Mean length: 346.64782755012203
Centerline extraction time consumption: 310.3029727935791ms 12601
/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W2tq9262/result_091

In [7]:

image = image_get(folders[0], 0)


/mnt/DATA/Mahsa/movies/LongRecordings/2024-09-17/W3sjr16/result_09172024185259/frame_0_label.tiff
Imgggg
[ 0 76]


In [10]:
img_index = 10
file_path = folders[0]
image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{img_index}.png"
    
image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)